# Congressional Voting Records: Data Preparation

## Overview

1. Load and clean the raw data by imputing missing values using mode imputation.
2. Create a single, hold-out test set.
3. From the main training set, generate multiple training datasets with varying **imbalance ratios (IR)** by undersampling the majority class.
4. For each IR, create **multiple repetitions** with different random samples of the minority class.
5. For each IR and repetition, create a size-matched **control dataset** with the original class ratio.
6. Preprocess and save all generated datasets.

## Dataset Information

- **Instances:** 435 congressmen (267 Democrats, 168 Republicans)
- **Features:** 16 binary voting issues
- **Target:** Party affiliation (0 = Democrat, 1 = Republican)
- **Natural Imbalance Ratio:** ~1.59:1 (Democrats are majority)

# Dataset Configuration

In [11]:
DATASET_NAME = "vote"

print(f"Dataset: {DATASET_NAME}")

Dataset: vote


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from pathlib import Path
import sys

# Add project root to path
sys.path.append(str(Path("../../").resolve()))
from config.config import get_config

config = get_config()

# Setup paths
RAW_PATH = Path("../../data/raw/vote.csv")
PROCESSED_PATH = Path(f"../../data/processed/{DATASET_NAME}/")

TARGET_FEATURE = "target"
CLASS_DEMOCRAT = 0   # Majority class
CLASS_REPUBLICAN = 1  # Minority class

RANDOM_STATE = config.experiment.random_state
IMBALANCE_RATIOS = config.experiment.imbalance_ratios
N_REPETITIONS = config.experiment.n_repetitions

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  - Dataset: {DATASET_NAME}")
print(f"  - Raw data path: {RAW_PATH}")
print(f"  - Processed data path: {PROCESSED_PATH}")
print(f"  - Target feature: {TARGET_FEATURE}")
print(f"\nLoading the raw dataset...")

df_raw = pd.read_csv(RAW_PATH)
# Clean column names
df_raw.columns = df_raw.columns.str.replace("'", "").str.strip()


Configuration:
  - Dataset: vote
  - Raw data path: ../../data/raw/vote.csv
  - Processed data path: ../../data/processed/vote
  - Target feature: target

Loading the raw dataset...
Raw dataset loaded. Shape: (435, 17)

First 5 rows:
   handicapped-infants  water-project-cost-sharing  \
0                  0.0                         1.0   
1                  0.0                         1.0   
2                  NaN                         1.0   
3                  0.0                         1.0   
4                  1.0                         1.0   

   adoption-of-the-budget-resolution  physician-fee-freeze  el-salvador-aid  \
0                                0.0                   1.0              1.0   
1                                0.0                   1.0              1.0   
2                                1.0                   NaN              1.0   
3                                1.0                   0.0              NaN   
4                                1.0          

# 2. Data Cleaning: Handle Missing Values

The Congressional Voting Records dataset contains missing values (represented as NaN) across multiple columns. Since all features are binary (0 = no, 1 = yes), we'll use **mode imputation** to fill missing values with the most frequent value for each feature.

In [13]:
print("Missing values per column before imputation:")
missing_before = df_raw.isnull().sum()
print(missing_before[missing_before > 0].sort_values(ascending=False))
print(f"\nTotal missing values: {df_raw.isnull().sum().sum()}")

Missing values per column before imputation:
export-administration-act-south-africa    104
water-project-cost-sharing                 48
education-spending                         31
duty-free-exports                          28
superfund-right-to-sue                     25
mx-missile                                 22
synfuels-corporation-cutback               21
crime                                      17
el-salvador-aid                            15
aid-to-nicaraguan-contras                  15
anti-satellite-test-ban                    14
handicapped-infants                        12
adoption-of-the-budget-resolution          11
physician-fee-freeze                       11
religious-groups-in-schools                11
immigration                                 7
dtype: int64

Total missing values: 392


In [14]:
print("Starting data cleaning...")
df_cleaned = df_raw.copy()

# Get feature columns (all except target)
feature_cols = [col for col in df_cleaned.columns if col != TARGET_FEATURE]

# Impute missing values with mode for each feature
total_imputed = 0
for col in feature_cols:
    missing_count = df_cleaned[col].isnull().sum()
    if missing_count > 0:
        mode_value = df_cleaned[col].mode()[0]
        df_cleaned[col] = df_cleaned[col].fillna(mode_value)
        total_imputed += missing_count
        print(f"  Imputed {missing_count} missing values in '{col}' with mode={mode_value}")

print(f"\nTotal values imputed: {total_imputed}")
print(f"Data cleaned. Shape: {df_cleaned.shape}")
print(f"Missing values after cleaning: {df_cleaned.isnull().sum().sum()}")

Starting data cleaning...
  Imputed 12 missing values in 'handicapped-infants' with mode=0.0
  Imputed 48 missing values in 'water-project-cost-sharing' with mode=1.0
  Imputed 11 missing values in 'adoption-of-the-budget-resolution' with mode=1.0
  Imputed 11 missing values in 'physician-fee-freeze' with mode=0.0
  Imputed 15 missing values in 'el-salvador-aid' with mode=1.0
  Imputed 11 missing values in 'religious-groups-in-schools' with mode=1.0
  Imputed 14 missing values in 'anti-satellite-test-ban' with mode=1.0
  Imputed 15 missing values in 'aid-to-nicaraguan-contras' with mode=1.0
  Imputed 22 missing values in 'mx-missile' with mode=1.0
  Imputed 7 missing values in 'immigration' with mode=1.0
  Imputed 21 missing values in 'synfuels-corporation-cutback' with mode=0.0
  Imputed 31 missing values in 'education-spending' with mode=0.0
  Imputed 25 missing values in 'superfund-right-to-sue' with mode=1.0
  Imputed 17 missing values in 'crime' with mode=1.0
  Imputed 28 missing 

# 3. Confirming Majority and Minority Classes

For our imbalance experiments:
- **Class 0 (Majority):** Democrat (267 congressmen, ~61.4%)
- **Class 1 (Minority):** Republican (168 congressmen, ~38.6%)
- **Natural Imbalance Ratio:** ~1.59:1

In [15]:
print("Target variable distribution:")
print(df_cleaned[TARGET_FEATURE].value_counts().sort_index())

n_majority = df_cleaned[TARGET_FEATURE].value_counts()[CLASS_DEMOCRAT]
n_minority = df_cleaned[TARGET_FEATURE].value_counts()[CLASS_REPUBLICAN]

print(f"\nClass balance: {n_majority} majority (Democrat), {n_minority} minority (Republican)")
print(f"Natural imbalance ratio: {n_majority / n_minority:.2f}:1")

Target variable distribution:
target
0    267
1    168
Name: count, dtype: int64

Class balance: 267 majority (Democrat), 168 minority (Republican)
Natural imbalance ratio: 1.59:1


# 4. Create a Hold-Out Test Set

We perform a one-time stratified split to create a final test set. All experimental datasets will be generated from the `train_full_df`.

In [16]:
X = df_cleaned.drop(TARGET_FEATURE, axis=1)
y = df_cleaned[[TARGET_FEATURE]]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=config.experiment.test_size,
    random_state=RANDOM_STATE,
    stratify=y
)

train_full_df = pd.concat([X_train_full, y_train_full], axis=1)

print(f"Full training set shape: {train_full_df.shape}")
print(f"Hold-out test set shape: {X_test.shape}")
print(f"\nTraining set class distribution:")
print(train_full_df[TARGET_FEATURE].value_counts().sort_index())

Full training set shape: (348, 17)
Hold-out test set shape: (87, 16)

Training set class distribution:
target
0    214
1    134
Name: count, dtype: int64


# 5. Generate Imbalanced and Control Datasets with Multiple Repetitions

For each IR, we create multiple repetitions by sampling different subsets of the minority class. This allows us to test whether methods work reliably across different minority class samples.

**Methodology:**
1. Start with the **full majority class** (Democrat).
2. **Undersample the minority class** (Republican) to achieve the desired Imbalance Ratio (IR).
3. **Repeat this sampling N_REPETITIONS times** with different random seeds.
4. Create a size-matched **control dataset** for each IR and repetition.

In [17]:
minority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_REPUBLICAN]  # Republican
majority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_DEMOCRAT]    # Democrat
n_minority_available = len(minority_df)
n_majority_available = len(majority_df)

print(f"\nFull training set composition: {n_majority_available} majority (Democrat), {n_minority_available} minority (Republican).")
print(f"\nGenerating datasets with {N_REPETITIONS} repetitions per imbalance ratio...")

generated_datasets = {}

for ir in IMBALANCE_RATIOS:
    print(f"Processing Imbalance Ratio (IR) = {ir}:1")
    
    for rep_id in range(1, N_REPETITIONS + 1):
        print(f"\n  Repetition {rep_id}/{N_REPETITIONS}")
        
        # Use different random seed for each repetition
        rep_seed = RANDOM_STATE + (ir * 1000) + rep_id
        
        if ir == 1:
            # For IR=1:1, undersample majority to match minority
            majority_undersampled = resample(
                majority_df,
                replace=False,
                n_samples=n_minority_available, 
                random_state=rep_seed 
            )
            imbalanced_df = pd.concat([majority_undersampled, minority_df])
            
        else:
            # For IR > 1, keep full majority and undersample minority
            majority_full_set = majority_df
            
            n_minority_imbalanced = int(n_majority_available / ir)

            if n_minority_imbalanced > n_minority_available:
                print(f"    SKIPPING: Cannot create {ir}:1 ratio as it requires more minority samples than available.")
                continue
            if n_minority_imbalanced < 1:
                print(f"    SKIPPING: Ratio {ir}:1 results in zero minority samples.")
                continue

            minority_undersampled = resample(
                minority_df,
                replace=False,
                n_samples=n_minority_imbalanced,
                random_state=rep_seed 
            )

            imbalanced_df = pd.concat([majority_full_set, minority_undersampled])

        total_size = len(imbalanced_df)
        
        dataset_key = f'imbalanced_ir_{ir}_rep{rep_id}'
        generated_datasets[dataset_key] = imbalanced_df
        
        n_maj = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_DEMOCRAT])
        n_min = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_REPUBLICAN])
        print(f"    Imbalanced set created: {total_size} samples ({n_maj} majority, {n_min} minority)")

        # Create control dataset with original class ratio
        if total_size >= len(train_full_df):
            control_df = train_full_df.copy()
        else:
            control_df, _ = train_test_split(
                train_full_df,
                train_size=total_size,
                random_state=rep_seed,  
                stratify=train_full_df[TARGET_FEATURE]
            )
        
        control_key = f'control_ir_{ir}_rep{rep_id}'
        generated_datasets[control_key] = control_df
        print(f"    Control set created:      {len(control_df)} samples (original class ratio)")

print(f"Total datasets created: {len(generated_datasets)}")
print(f"  - Imbalanced: {len([k for k in generated_datasets.keys() if 'imbalanced' in k])}")
print(f"  - Control: {len([k for k in generated_datasets.keys() if 'control' in k])}")


Full training set composition: 214 majority (Democrat), 134 minority (Republican).

Generating datasets with 10 repetitions per imbalance ratio...
Processing Imbalance Ratio (IR) = 1:1

  Repetition 1/10
    Imbalanced set created: 268 samples (134 majority, 134 minority)
    Control set created:      268 samples (original class ratio)

  Repetition 2/10
    Imbalanced set created: 268 samples (134 majority, 134 minority)
    Control set created:      268 samples (original class ratio)

  Repetition 3/10
    Imbalanced set created: 268 samples (134 majority, 134 minority)
    Control set created:      268 samples (original class ratio)

  Repetition 4/10
    Imbalanced set created: 268 samples (134 majority, 134 minority)
    Control set created:      268 samples (original class ratio)

  Repetition 5/10
    Imbalanced set created: 268 samples (134 majority, 134 minority)
    Control set created:      268 samples (original class ratio)

  Repetition 6/10
    Imbalanced set created: 26

    Control set created:      285 samples (original class ratio)

  Repetition 3/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 4/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 5/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 6/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 7/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 8/10
    Imbalanced set created: 285 samples (214 majority, 71 minority)
    Control set created:      285 samples (original class ratio)

  Repetition 9/10
    

# 6. Preprocessing and Saving All Datasets

All features are binary (0 or 1). We apply StandardScaler for consistency with other datasets in the experiment pipeline.

We fit the scaler **once** on the full training data. Then, we transform all generated training sets and the hold-out test set using this single, consistent scaler.

In [18]:
FEATURES_TO_SCALE = [col for col in df_cleaned.columns if col != TARGET_FEATURE]

print(f"Features to scale ({len(FEATURES_TO_SCALE)}):")
for i, feat in enumerate(FEATURES_TO_SCALE, 1):
    print(f"  {i}. {feat}")

Features to scale (16):
  1. handicapped-infants
  2. water-project-cost-sharing
  3. adoption-of-the-budget-resolution
  4. physician-fee-freeze
  5. el-salvador-aid
  6. religious-groups-in-schools
  7. anti-satellite-test-ban
  8. aid-to-nicaraguan-contras
  9. mx-missile
  10. immigration
  11. synfuels-corporation-cutback
  12. education-spending
  13. superfund-right-to-sue
  14. crime
  15. duty-free-exports
  16. export-administration-act-south-africa


In [19]:
scaler = StandardScaler()
scaler.fit(X_train_full[FEATURES_TO_SCALE])

print("Scaling and saving datasets...\n")

for name, df in generated_datasets.items():
    X_temp = df.drop(columns=[TARGET_FEATURE])
    y_temp = df[[TARGET_FEATURE]]

    X_processed = scaler.transform(X_temp[FEATURES_TO_SCALE])
    X_processed_df = pd.DataFrame(X_processed, columns=FEATURES_TO_SCALE)
    
    final_df = X_processed_df.reset_index(drop=True)
    final_df[TARGET_FEATURE] = y_temp.reset_index(drop=True)
    
    save_path = PROCESSED_PATH / f"train_{name}.csv"
    final_df.to_csv(save_path, index=False)
    print(f"Saved: {save_path.name}")

# Process and save test set
X_test_processed = scaler.transform(X_test[FEATURES_TO_SCALE])
X_test_processed_df = pd.DataFrame(X_test_processed, columns=FEATURES_TO_SCALE)

test_df = X_test_processed_df.reset_index(drop=True)
test_df[TARGET_FEATURE] = y_test.reset_index(drop=True)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print(f"\nSaved test set: test.csv")
print(f"Total training files: {len(generated_datasets)}")

Scaling and saving datasets...

Saved: train_imbalanced_ir_1_rep1.csv
Saved: train_control_ir_1_rep1.csv
Saved: train_imbalanced_ir_1_rep2.csv
Saved: train_control_ir_1_rep2.csv
Saved: train_imbalanced_ir_1_rep3.csv
Saved: train_control_ir_1_rep3.csv
Saved: train_imbalanced_ir_1_rep4.csv
Saved: train_control_ir_1_rep4.csv
Saved: train_imbalanced_ir_1_rep5.csv
Saved: train_control_ir_1_rep5.csv
Saved: train_imbalanced_ir_1_rep6.csv
Saved: train_control_ir_1_rep6.csv
Saved: train_imbalanced_ir_1_rep7.csv
Saved: train_control_ir_1_rep7.csv
Saved: train_imbalanced_ir_1_rep8.csv
Saved: train_control_ir_1_rep8.csv
Saved: train_imbalanced_ir_1_rep9.csv
Saved: train_control_ir_1_rep9.csv
Saved: train_imbalanced_ir_1_rep10.csv
Saved: train_control_ir_1_rep10.csv
Saved: train_imbalanced_ir_3_rep1.csv
Saved: train_control_ir_3_rep1.csv
Saved: train_imbalanced_ir_3_rep2.csv
Saved: train_control_ir_3_rep2.csv
Saved: train_imbalanced_ir_3_rep3.csv
Saved: train_control_ir_3_rep3.csv
Saved: train_imba

In [20]:
import json
from datetime import datetime

# Save metadata about this dataset processing
metadata = {
    "dataset_name": DATASET_NAME,
    "target_feature": TARGET_FEATURE,
    "processing_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_train_files": len(generated_datasets),
    "imbalance_ratios": IMBALANCE_RATIOS,
    "n_repetitions": N_REPETITIONS,
    "random_state": RANDOM_STATE,
    "test_size": len(test_df),
    "features": FEATURES_TO_SCALE,
    "class_mapping": {
        "0": "Democrat (majority)",
        "1": "Republican (minority)"
    },
    "notes": "1984 Congressional Voting Records - predicting party affiliation from 16 key votes"
}

metadata_path = PROCESSED_PATH / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to: {metadata_path}")
print("\nProcessing complete! All datasets are ready for experiments.")


Metadata saved to: ../../data/processed/vote/metadata.json

Processing complete! All datasets are ready for experiments.
